In [5]:
REPO = "https://github.com/punith24246/hiver-support-agent.git"

import os, sys, subprocess
if not os.path.exists("hiver-support-agent"):
    subprocess.run(["git","clone",REPO], check=True)
os.chdir("/content/hiver-support-agent")
!pip install -q -r requirements.txt
sys.path.insert(0, os.getcwd())
print(os.getcwd())

ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'
/content/hiver-support-agent


In [6]:
import shutil, os, sys, subprocess

os.chdir("/content")
if os.path.exists("hiver-support-agent"):
    shutil.rmtree("hiver-support-agent")

REPO = "https://github.com/punith24246/hiver-support-agent.git"
subprocess.run(["git", "clone", REPO], check=True)
os.chdir("/content/hiver-support-agent")

print(os.getcwd())
print(os.listdir())

/content/hiver-support-agent
['.git', 'hiver-support-agent']


In [7]:
os.chdir("hiver-support-agent")
print(os.getcwd())
print(os.listdir())

/content/hiver-support-agent/hiver-support-agent
['support_agent', 'data', 'docs', 'README.md', '.gitignore', 'runs', 'configs', 'notebooks', 'tests', 'evaluation', 'requirements.txt', 'scripts']


In [8]:
!pip install -q -r requirements.txt
sys.path.insert(0, os.getcwd())
!AGENT_MOCK=1 python -m pytest -q

...........                                                              [100%]
11 passed in 2.53s


In [9]:
from google.colab import userdata
os.environ["GROQ_API_KEY"]    = userdata.get("GROQ_API_KEY")
os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"]      = userdata.get("KAGGLE_KEY")
print("keys loaded ok")

keys loaded ok


In [10]:
!python -m scripts.get_data

100% 169M/169M [00:01<00:00, 90.3MB/s]
Extracting files...
-> data/twcs.csv (517 MB)


In [11]:
import pandas as pd
df = pd.read_csv("data/twcs.csv", nrows=200_000)
print(df[df.author_id=="Ask_Spectrum"].shape, "Spectrum tweets in first 200k rows")
df.head(3)

(1088, 7) Spectrum tweets in first 200k rows


,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id
0,1,sprintcare,False,Tue Oct 31 22:10:47 +0000 2017,@115712 I understand. I would like to assist y...,2,3.0
1,2,115712,True,Tue Oct 31 22:11:45 +0000 2017,@sprintcare and how do you propose we do that,NaN,1.0
2,3,115712,True,Tue Oct 31 22:08:27 +0000 2017,@sprintcare I have sent several private messag...,1,4.0


In [12]:
!python -m scripts.build_golden_pool --config configs/spectrum.yaml --n 220 --blind 50 --prelabel
!head -c 800 data/golden/pool.jsonl

eval pool (time-held-out): 6,041 messages
pre-labelling 170 examples with openai/gpt-oss-120b ...

wrote 220 examples -> data/golden/to_label.jsonl
  stratified: 128   random: 92   blind arm: 50

Next: python -m scripts.label_cli
head: cannot open 'data/golden/pool.jsonl' for reading: No such file or directory


In [13]:
!head -c 800 data/golden/to_label.jsonl

{"customer_tweet_id": 2816599, "text": "any reports of internet outages in the Dallas area? Ours is not working at all \ud83e\udd28", "brand_reply_actual": "My apologies for the interruption in your service Kenneth. If you can DM me and provide me with the full address, phone number and/or account number I would be glad to look into that for you. ^TV", "sample_arm": "random", "blind": true, "prelabel_intent": null, "intent": null, "action": null, "action_reason": null}
{"customer_tweet_id": 2211874, "text": "I just want to know why my bill continues to escalate MORE AND MORE- DESPITE NO IMPROVEMENT IN SERVICE!", "brand_reply_actual": "Please be sure to send this information via DM. ^MO", "sample_arm": "random", "blind": true, "prelabel_intent": null, "intent": null, "action": null, "action

In [22]:

import json, os
from support_agent.intents import INTENTS

POOL, OUT = "data/golden/to_label.jsonl", "data/golden/golden.jsonl"
load = lambda p: [json.loads(l) for l in open(p)] if os.path.exists(p) else []
pool, done = load(POOL), {r["customer_tweet_id"] for r in load(OUT)}
todo = [r for r in pool if r["customer_tweet_id"] not in done]
names = [i.name for i in INTENTS]

print(f"{len(done)} already labelled, {len(todo)} to go.\n")
print("Intents:", {i+1: n for i, n in enumerate(names)})
print("Action: a = auto, e = escalate\n")

for idx, r in enumerate(todo):
    print("="*70)
    print(f"[{idx+1}/{len(todo)}] arm={r.get('sample_arm')} blind={r.get('blind', False)}")
    print(r["text"])
    if r.get("brand_reply_actual"):
        print("actual brand reply:", r["brand_reply_actual"])
    if not r.get("blind") and r.get("prelabel_intent"):
        print("pre-label:", r["prelabel_intent"], "  [enter to accept]")
    print("-"*70)

    raw = input("intent (number, or enter to accept pre-label): ").strip()
    if raw == "" and r.get("prelabel_intent"):
        intent = r["prelabel_intent"]
    else:
        intent = names[int(raw)-1]

    act = input("action (a/e): ").strip().lower()
    action = "auto" if act == "a" else "escalate"
    reason = input("reason (optional, enter to skip): ").strip()

    rec = dict(r, intent=intent, action=action, action_reason=reason)
    with open(OUT, "a") as f:
        f.write(json.dumps(rec) + "\n")

    cont = input("\n[enter] next, 'q' to stop: ").strip()
    if cont.lower() == "q":
        break



print("\nStopped. Progress saved in", OUT)


44 already labelled, 176 to go.

Intents: {1: 'outage', 2: 'connectivity_degraded', 3: 'billing', 4: 'account_access', 5: 'equipment', 6: 'appointment', 7: 'channel_content', 8: 'cancellation', 9: 'other'}
Action: a = auto, e = escalate

[1/176] arm=stratified blind=True
is there a known outage at zip code 10034 right now?
actual brand reply: I can look into that for you. Can you please send me your account or phone number directly in a private message? ^AS
----------------------------------------------------------------------
intent (number, or enter to accept pre-label): 1
action (a/e): a
reason (optional, enter to skip): 

[enter] next, 'q' to stop: 
[2/176] arm=random blind=True
my internet appears down after I've unplugged and rebooted several times. Are there currently outages in the Dallas, TX area?
actual brand reply: Hello. I apologize for the service interruption and I would like to check on this for you. Please send the phone number associated with your account in a DM for f

In [25]:
!python -m evaluation.run_eval --config configs/spectrum.yaml --limit 25 --no-judge

[1/6] loading brand corpus ...
      grounding corpus: 18,123 historical pairs
[2/6] golden set: 25 labelled examples
[3/6] running agent on 13 held-out examples ...
[4/6] scoring intent + routing ...
[6/6] writing artefacts ...

Done -> runs/20260917-134056/summary.md
# Results (n = 13 held-out golden examples)

| system | intent macro-F1 | escalation recall | false-auto rate | automation rate | judge overall |
|---|---|---|---|---|---|
| B0_trivial | 0.062 | 0.000 | 1.000 | 1.000 | nan |
| B1_simple | 0.074 | 1.000 | 0.000 | 0.000 | nan |
| agent | 0.335 | 0.875 | 0.125 | 0.154 | nan |



In [32]:
import os
os.environ["AGENT_MOCK"] = "1"

In [34]:
os.environ["AGENT_MOCK"] = "0"

In [ ]:
Intents: {1: 'outage', 2: 'connectivity_degraded', 3: 'billing', 4: 'account_access', 5: 'equipment', 6: 'appointment', 7: 'channel_content', 8: 'cancellation', 9: 'other'}
Action: a = auto, e = escalate


In [ ]:
"""Thin LLM layer.

Three things matter here and nothing else:
  * Disk cache keyed on (model, prompt). Without it the 15-minute reproduce
    claim is a lie on the second run, and evaluation becomes non-deterministic.
  * Defensive JSON parsing. Small open models emit fenced JSON, trailing prose,
    and occasionally a leading "Here is the JSON:". We strip all of it.
  * An offline mock so `pytest` and the smoke test run with no API key.
"""

from __future__ import annotations

import hashlib
import json
import os
import re
import sqlite3
import threading
import time
from typing import Any

_CACHE_LOCK = threading.Lock()
_CACHE_PATH = os.environ.get("AGENT_CACHE", "runs/llm_cache.sqlite")


def _cache_conn():
    os.makedirs(os.path.dirname(_CACHE_PATH) or ".", exist_ok=True)
    conn = sqlite3.connect(_CACHE_PATH, check_same_thread=False)
    conn.execute("CREATE TABLE IF NOT EXISTS cache (k TEXT PRIMARY KEY, v TEXT)")
    return conn


_CONN = _cache_conn()


def _key(model: str, system: str, user: str, temperature: float) -> str:
    raw = json.dumps([model, system, user, temperature], sort_keys=True)
    return hashlib.sha256(raw.encode()).hexdigest()


def cache_get(k: str):
    with _CACHE_LOCK:
        row = _CONN.execute("SELECT v FROM cache WHERE k=?", (k,)).fetchone()
    return row[0] if row else None


def cache_put(k: str, v: str):
    with _CACHE_LOCK:
        _CONN.execute("INSERT OR REPLACE INTO cache VALUES (?,?)", (k, v))
        _CONN.commit()


# ---------------------------------------------------------------------------

class LLM:
    def __init__(self, model: str, temperature: float = 0.0, mock: bool = False):
        self.model = model
        self.temperature = temperature
        self.mock = mock or os.environ.get("AGENT_MOCK") == "1"
        self._client = None

    @property
    def client(self):
        if self._client is None:
            from openai import OpenAI  # Groq is OpenAI-API compatible
            self._client = OpenAI(
                api_key=os.environ["GROQ_API_KEY"],
                base_url="https://api.groq.com/openai/v1",
            )
        return self._client

    def complete(self, system: str, user: str, max_tokens: int = 700) -> str:
        if self.mock:
            return _mock_response(system, user)

        k = _key(self.model, system, user, self.temperature)
        hit = cache_get(k)
        if hit is not None:
            return hit

        last_err = None
        for attempt in range(4):
            try:
                resp = self.client.chat.completions.create(
                    model=self.model,
                    temperature=self.temperature,
                    max_tokens=max_tokens,
                    messages=[
                        {"role": "system", "content": system},
                        {"role": "user", "content": user},
                    ],
                )
                out = resp.choices[0].message.content or ""
                cache_put(k, out)
                return out
            except Exception as e:  # rate limits, transient 5xx
                last_err = e
                time.sleep(2 ** attempt)
        raise RuntimeError(f"LLM call failed after retries: {last_err}")

    def complete_json(self, system: str, user: str, max_tokens: int = 700) -> dict[str, Any]:
        raw = self.complete(system, user, max_tokens=max_tokens)
        return parse_json(raw)


_FENCE = re.compile(r"```(?:json)?\s*(.*?)```", re.S)


def parse_json(raw: str) -> dict[str, Any]:
    """Extract the first JSON object from a possibly chatty completion."""
    if not raw:
        return {}
    m = _FENCE.search(raw)
    if m:
        raw = m.group(1)
    start = raw.find("{")
    if start == -1:
        return {}
    depth, end = 0, None
    for i, ch in enumerate(raw[start:], start):
        if ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                end = i + 1
                break
    if end is None:
        return {}
    try:
        return json.loads(raw[start:end])
    except json.JSONDecodeError:
        # last resort: single -> double quotes
        try:
            return json.loads(raw[start:end].replace("'", '"'))
        except Exception:
            return {}


def _mock_response(system: str, user: str) -> str:
    """Deterministic stand-in so the pipeline is testable with no network."""
    if "judge" in system.lower():
        return json.dumps({
            "groundedness": 3, "helpfulness": 3, "tone": 3,
            "safety": 5, "overall": 3, "rationale": "mock",
        })
    if "intent" in system.lower():
        return json.dumps({
            "intent": "outage", "confidence": 0.5,
            "rationale": "mock", "severity": "medium",
        })
    return json.dumps({"reply": "Sorry for the trouble — can you DM us your address?"})
